# 03 — Split por Ator (Actor Hold-Out)

**Objetivo:** Criar split train/test baseado em atores para evitar data leakage.
O mesmo ator nunca aparece em train e test simultaneamente.

**Padrão sugerido:** Atores 01–02 = test, Atores 03–24 = train.

**Output:**
- `data/ravdess_landmarks_kaggle/02_splits/split_actor_holdout.json`
- Tabelas de distribuição por classe em cada split

In [ ]:
import sys, os, json

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.ravdess_utils import EMOTION_MAP, EMOTION_LABELS

print(f"Raiz: {ROOT}")

## 3.1 Carregar Dataset T=100

In [ ]:
DATA_DIR = os.path.join(ROOT, 'data', 'ravdess_landmarks_kaggle', '01_processed_T100')
SPLIT_DIR = os.path.join(ROOT, 'data', 'ravdess_landmarks_kaggle', '02_splits')
os.makedirs(SPLIT_DIR, exist_ok=True)

data = np.load(os.path.join(DATA_DIR, 'dataset_T100.npz'), allow_pickle=True)
X = data['X']
y = data['y']
actor_ids = data['actor_ids']

print(f"X: {X.shape}, y: {y.shape}, actor_ids: {actor_ids.shape}")
print(f"Atores únicos: {sorted(np.unique(actor_ids))}")
print(f"Classes únicas: {sorted(np.unique(y))}")

## 3.2 Definir Split por Ator

Usamos atores 1 e 2 como **test** (1 masculino, 1 feminino no RAVDESS)
e atores 3–24 como **train**.

In [ ]:
# Definir atores de test
TEST_ACTORS = [1, 2]
all_actors = sorted(np.unique(actor_ids).tolist())
TRAIN_ACTORS = [a for a in all_actors if a not in TEST_ACTORS]

print(f"Atores TEST:  {TEST_ACTORS}")
print(f"Atores TRAIN: {TRAIN_ACTORS}")

# Criar máscaras
train_mask = np.isin(actor_ids, TRAIN_ACTORS)
test_mask = np.isin(actor_ids, TEST_ACTORS)

# Índices
train_indices = np.where(train_mask)[0].tolist()
test_indices = np.where(test_mask)[0].tolist()

print(f"\nTrain: {len(train_indices)} amostras")
print(f"Test:  {len(test_indices)} amostras")
print(f"Total: {len(train_indices) + len(test_indices)} (deve ser {len(y)})")

## 3.3 Distribuição de Classes por Split

In [ ]:
# Tabela de distribuição
y_train = y[train_mask]
y_test = y[test_mask]

rows = []
for code in sorted(np.unique(y)):
    label = EMOTION_MAP.get(code + 1, f"emo_{code}")
    n_train = (y_train == code).sum()
    n_test = (y_test == code).sum()
    rows.append({
        'emotion_code': code,
        'emotion_label': label,
        'train': int(n_train),
        'test': int(n_test),
        'total': int(n_train + n_test),
        'train_pct': round(100 * n_train / (n_train + n_test), 1) if (n_train + n_test) > 0 else 0,
    })

# Totais
rows.append({
    'emotion_code': -1,
    'emotion_label': 'TOTAL',
    'train': len(y_train),
    'test': len(y_test),
    'total': len(y),
    'train_pct': round(100 * len(y_train) / len(y), 1),
})

split_table = pd.DataFrame(rows)
print("\nDistribuição de Classes por Split:")
print("=" * 70)
split_table

In [ ]:
# Visualização
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

data_plot = split_table[split_table['emotion_code'] >= 0]
x_pos = np.arange(len(data_plot))
w = 0.35

# Barplot side-by-side
axes[0].bar(x_pos - w/2, data_plot['train'], w, label='Train', color='steelblue')
axes[0].bar(x_pos + w/2, data_plot['test'], w, label='Test', color='coral')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(data_plot['emotion_label'], rotation=45, ha='right')
axes[0].set_ylabel('Contagem')
axes[0].set_title('Distribuição de Classes: Train vs Test')
axes[0].legend()

# Proporções
axes[1].bar(x_pos, data_plot['train_pct'], color='steelblue')
axes[1].axhline(y=100 * len(y_train) / len(y), color='red', linestyle='--', label=f'Média={100*len(y_train)/len(y):.0f}%')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(data_plot['emotion_label'], rotation=45, ha='right')
axes[1].set_ylabel('% Train')
axes[1].set_title('Proporção Train por Classe')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3.4 Salvar Split JSON

In [ ]:
split_info = {
    'description': 'Actor hold-out split para RAVDESS emotion classification',
    'test_actors': TEST_ACTORS,
    'train_actors': TRAIN_ACTORS,
    'train_indices': train_indices,
    'test_indices': test_indices,
    'n_train': len(train_indices),
    'n_test': len(test_indices),
    'n_total': len(y),
}

split_path = os.path.join(SPLIT_DIR, 'split_actor_holdout.json')
with open(split_path, 'w') as f:
    json.dump(split_info, f, indent=2)

print(f"Split salvo em: {split_path}")
print(f"Train: {len(train_indices)} amostras ({len(TRAIN_ACTORS)} atores)")
print(f"Test:  {len(test_indices)} amostras ({len(TEST_ACTORS)} atores)")

print("\nNotebook 03 concluído com sucesso!")